# Unit 2 Team Project — Build a Grammar-Driven Conversational Agent
### 21CSE552T · Computational Linguistics · Overview of Grammar

**Teams of 3–4. One notebook per team. Work top to bottom across the unit — each Part is added after the corresponding lecture.**

---

## What you are building

A working conversational agent for a domain you choose, where **every layer of the agent is one topic from Unit 2**:

| Part | Unit 2 topic | What you build |
|---|---|---|
| 1 | (setup) | Your domain + a 20-utterance test set |
| 2 | A Simple Context-Free Grammar | The CFG that recognises user utterances |
| 3 | Transformational Grammars | Statement ↔ question transformations for confirmations |
| 4 | Valencies and Interpretation | Verb valency tables → **intent schemas** |
| 5 | Constraints | Feature agreement that blocks what the CFG overgenerates |
| 6 | HPSG and Unification | `unify()` → **multi-turn dialogue state tracking** |
| 7 | MTT: Government Patterns | GP tables → the bilingual **response generator** |
| 8 | MTT: Dependency Trees | spaCy/Stanza slot extraction vs. your hand-built CFG |
| 9 | Capstone | Assemble, evaluate, and compare against an LLM |

**The punchline of the unit:** a subcategorization frame *is* an intent schema, unification *is* dialogue state tracking, and a government pattern *is* your NLG template. You are rebuilding a modern dialogue system out of 1957–1980 linguistic theory, and then finding out exactly where it wins and where it loses.

## Rules

1. The **canteen** domain is fully worked out as a reference implementation. **Do not edit it.** Copy the pattern into the `# ==== TODO (TEAM) ====` cells for *your* domain.
2. Every part must run end to end before you move on. A broken Part 2 breaks Parts 5, 8 and 9.
3. Bilingual requirement: your bot must confirm orders in **English and one regional language** (Tamil / Hindi / Malayalam / Telugu / Kannada / Marathi / Bengali …). At least one team member must be a fluent speaker — you are responsible for the correctness of that language's data.
4. Keep your test set honest. You will hand it to another team in Part 9, and receive theirs.

## Marks (20, CIA)

| Component | Marks |
|---|---|
| Working bot (Parts 2–8 all run) | 6 |
| Correctness of valency / GP / feature tables | 4 |
| Quality of your adversarial test set | 4 |
| Comparative analysis write-up (Part 9) | 4 |
| Lightning demo | 2 |


---
# Part 0 — Setup

Runs on Google Colab or a local Jupyter/Anaconda install. Only Part 8 needs a download.


In [1]:
# Core install. On Colab this takes ~20 s. Locally, nltk is usually already present.
%pip install -q nltk


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json, re, itertools, sys
from collections import defaultdict

import nltk
from nltk import CFG
from nltk.parse import ChartParser
from nltk.grammar import FeatureGrammar
from nltk.parse import FeatureChartParser
from nltk.parse.generate import generate

print("python :", sys.version.split()[0])
print("nltk   :", nltk.__version__)


python : 3.11.9
nltk   : 3.9.1


In [3]:
# ==== TODO (TEAM) ====  Fill this in first. It is checked in the demo.
TEAM = {
    "team_id": "T00",
    "members": ["Name 1 (RegNo)", "Name 2 (RegNo)", "Name 3 (RegNo)"],
}

# Pick ONE:  "canteen" | "helpdesk" | "travel" | "lab"
#   canteen  – campus canteen / mess ordering
#   helpdesk – hostel & library helpdesk (book renewal, room complaint, gate pass)
#   travel   – train / bus seat booking
#   lab      – lab equipment issue & return
DOMAIN = "canteen"

# Your second language. Used in Parts 3, 5 and 7.
L2 = "Tamil"

print(TEAM["team_id"], "|", DOMAIN, "| English +", L2)


T00 | canteen | English + Tamil


---
# Part 1 — Your test set

Before writing a single grammar rule, write down what real users actually say. **This list is the single most reused artifact in the notebook** — Parts 2, 4, 8 and 9 all score against it.

Each entry has:
- `text` — the utterance, lowercase, no punctuation
- `intent` — the *verb* the user is performing (this becomes your intent label in Part 4)
- `slots` — what the utterance actually fills in

Write utterances the way people *speak*, not the way a grammar textbook writes them. Include at least:
- 3 with a quantity, 3 without
- 2 that are questions
- 2 that amend a previous turn ("make it three", "no, cancel that")
- 2 that are ungrammatical or elliptical but perfectly understandable


In [4]:
# ===== REFERENCE IMPLEMENTATION (canteen) — do not edit =====
CANTEEN_TESTSET = [
    {"text": "i want two dosas",              "intent": "order",  "slots": {"item": "dosa", "quantity": "two"}},
    {"text": "i want a masala dosa",          "intent": "order",  "slots": {"item": "dosa", "quantity": "one", "modifier": "masala"}},
    {"text": "give me one coffee",            "intent": "order",  "slots": {"item": "coffee", "quantity": "one"}},
    {"text": "can i get three idlis",         "intent": "order",  "slots": {"item": "idli", "quantity": "three"}},
    {"text": "i need two teas",               "intent": "order",  "slots": {"item": "tea", "quantity": "two"}},
    {"text": "add one coffee",                "intent": "order",  "slots": {"item": "coffee", "quantity": "one"}},
    {"text": "i want two dosas and one coffee","intent": "order", "slots": {"item": "dosa", "quantity": "two"}},
    {"text": "cancel my order",               "intent": "cancel", "slots": {}},
    {"text": "make it three",                 "intent": "amend",  "slots": {"quantity": "three"}},
    {"text": "what is the price of dosa",     "intent": "price",  "slots": {"item": "dosa"}},
    {"text": "how much for two idlis",        "intent": "price",  "slots": {"item": "idli", "quantity": "two"}},
    {"text": "is dosa available",             "intent": "avail",  "slots": {"item": "dosa"}},
]
print(len(CANTEEN_TESTSET), "reference utterances")


12 reference utterances


In [5]:
# ==== TODO (TEAM) ====  20 utterances for YOUR domain, annotated the same way.
TESTSET = [
    # {"text": "...", "intent": "...", "slots": {"...": "..."}},
]

# Use the reference set until you have written your own.
if not TESTSET:
    TESTSET = CANTEEN_TESTSET
    print("!! Using the reference canteen set. Replace TESTSET before the demo.")

assert len(TESTSET) >= 12, "You need at least 12 utterances (target 20)."
INTENTS = sorted({u["intent"] for u in TESTSET})
print(len(TESTSET), "utterances |", len(INTENTS), "intents:", INTENTS)


!! Using the reference canteen set. Replace TESTSET before the demo.
12 utterances | 5 intents: ['amend', 'avail', 'cancel', 'order', 'price']


In [6]:
def tokens(s):
    '''Normalise an utterance to a token list: lowercase, punctuation stripped.'''
    return re.findall(r"[a-z0-9']+", s.lower())

for u in TESTSET[:3]:
    print(tokens(u["text"]))


['i', 'want', 'two', 'dosas']
['i', 'want', 'a', 'masala', 'dosa']
['give', 'me', 'one', 'coffee']


---
# Part 2 — A Simple Context-Free Grammar
### Topic 2 · *Grammar Golf*

A CFG has **non-terminals** (phrase categories), **terminal rules** (category → word) and a **start symbol** `S`. Parsing asks: *can this string be derived from `S`?* Generating asks: *what strings can `S` produce?*

Two failure modes, and you are going to measure both on your own grammar:

- **Undergeneration** — a perfectly good utterance your grammar rejects.
- **Overgeneration** — nonsense your grammar happily accepts (`the dosa drinks a coffee`). This is Bolshakov & Gelbukh's *"why meaningless sentences appear"*, and Part 5 is the fix.

**Your score is `coverage ÷ number of rules`.** Fewest rules for full coverage wins. That is the golf.


In [7]:
# ===== REFERENCE IMPLEMENTATION (canteen) — do not edit =====
CANTEEN_CFG = CFG.fromstring('''
    S      -> ORDER | CANCEL | AMEND | PRICE | AVAIL
    ORDER  -> WANT NP | WANT NP 'and' NP
    CANCEL -> 'cancel' DET N
    AMEND  -> 'make' 'it' NUM
    PRICE  -> 'what' 'is' 'the' 'price' 'of' N | 'how' 'much' 'for' NP
    AVAIL  -> 'is' N 'available'
    NP     -> NUM N | DET N | NUM ADJ N | DET ADJ N | N
    WANT   -> 'i' 'want' | 'i' 'need' | 'give' 'me' | 'can' 'i' 'get' | 'add'
    NUM    -> 'one' | 'two' | 'three'
    DET    -> 'a' | 'my' | 'the'
    ADJ    -> 'masala' | 'plain'
    N      -> 'dosa' | 'dosas' | 'idli' | 'idlis' | 'coffee' | 'tea' | 'teas' | 'order'
''')
print(len(CANTEEN_CFG.productions()), "productions")


38 productions


In [8]:
def coverage(grammar, testset, verbose=True):
    '''How many utterances does this grammar parse? Returns (n_ok, failures).'''
    parser = ChartParser(grammar)
    ok, failures = 0, []
    for u in testset:
        toks = tokens(u["text"])
        try:
            trees = list(parser.parse(toks))
        except ValueError:          # token not in the grammar's lexicon at all
            trees = []
        if trees:
            ok += 1
        else:
            failures.append(u["text"])
    if verbose:
        print(f"coverage: {ok}/{len(testset)} = {ok/len(testset):.0%}")
        for f in failures:
            print("   MISS:", f)
    return ok, failures


def golf_score(grammar, testset):
    ok, _ = coverage(grammar, testset, verbose=False)
    n_rules = len(grammar.productions())
    print(f"covered {ok}/{len(testset)} with {n_rules} rules  ->  score = {ok/n_rules:.3f}")
    return ok / n_rules


coverage(CANTEEN_CFG, CANTEEN_TESTSET)
golf_score(CANTEEN_CFG, CANTEEN_TESTSET)


coverage: 12/12 = 100%
covered 12/12 with 38 rules  ->  score = 0.316


0.3157894736842105

In [9]:
# The constituency tree and the bracketed nested structure (slides 'The constituency tree')
p = ChartParser(CANTEEN_CFG)
tree = next(p.parse(tokens("i want two dosas")))
tree.pretty_print()
print(tree)          # bracketed form


                S                
                |                 
              ORDER              
      __________|________         
     |                   NP      
     |                ___|____    
    WANT            NUM       N  
  ___|____           |        |   
 i       want       two     dosas

(S (ORDER (WANT i want) (NP (NUM two) (N dosas))))


In [10]:
# ==== TODO (TEAM) ====  Your CFG. Aim for full coverage with as few rules as possible.
MY_CFG = CFG.fromstring('''
    S    -> WANT NP
    WANT -> 'i' 'want'
    NP   -> NUM N | N
    NUM  -> 'one' | 'two'
    N    -> 'thing' | 'things'
''')

coverage(MY_CFG, TESTSET)
golf_score(MY_CFG, TESTSET)


coverage: 0/12 = 0%
   MISS: i want two dosas
   MISS: i want a masala dosa
   MISS: give me one coffee
   MISS: can i get three idlis
   MISS: i need two teas
   MISS: add one coffee
   MISS: i want two dosas and one coffee
   MISS: cancel my order
   MISS: make it three
   MISS: what is the price of dosa
   MISS: how much for two idlis
   MISS: is dosa available
covered 0/12 with 8 rules  ->  score = 0.000


0.0

In [11]:
# --- Overgeneration probe: what CAN your grammar say? ---
# Read this list and mark every string that is nonsense. That is your overgeneration rate.
gen = list(generate(CANTEEN_CFG, depth=6, n=25))
for s in gen:
    print(" ", " ".join(s))
print(f"\n{len(gen)} sample strings generated (depth-limited).")


  i want one dosa
  i want one dosas
  i want one idli
  i want one idlis
  i want one coffee
  i want one tea
  i want one teas
  i want one order
  i want two dosa
  i want two dosas
  i want two idli
  i want two idlis
  i want two coffee
  i want two tea
  i want two teas
  i want two order
  i want three dosa
  i want three dosas
  i want three idli
  i want three idlis
  i want three coffee
  i want three tea
  i want three teas
  i want three order
  i want a dosa

25 sample strings generated (depth-limited).


In [12]:
# ==== TODO (TEAM) ====  ATTACK LOG — you fill this in during the in-class grammar swap.
ATTACKS = {
    # sensible utterances the grammar WRONGLY REJECTS
    "undergeneration": [
        # "two dosa please",
    ],
    # nonsense the grammar WRONGLY ACCEPTS
    "overgeneration": [
        # "the order drinks a masala",
    ],
}
print("under:", len(ATTACKS["undergeneration"]), " over:", len(ATTACKS["overgeneration"]))


under: 0  over: 0


---
# Part 3 — Transformational Grammars
### Topic 3 · *Deep structure → surface structure*

A CFG generates a declarative. Chomsky's move: keep **one** deep structure, then apply ordered **transformations** to derive the surface forms — questions, passives, negatives.

Your bot needs exactly this. It understands `i want two dosas` (declarative), and must confirm with `do you want two dosas?` (interrogative). Same meaning, different surface string.

Two things to observe, both of which are on the slides:
1. **Transformations work like a program** — they apply in a fixed order, and the wrong order produces garbage. Run the cell and see it.
2. Your **regional language does it differently.** Tamil/Hindi form yes-no questions with a particle or clitic (Tamil `-ā`, Hindi `kyā`), *not* by moving the auxiliary to the front. Movement is not a language universal — it is an English fact. Record that in the last cell.


In [13]:
# A deep structure: a flat dict of grammatical functions.
def deep_structure(subj, verb, obj, aux=None):
    return {"subj": list(subj), "aux": list(aux) if aux else [], "verb": list(verb), "obj": list(obj)}

def linearize(d):
    return " ".join(d["subj"] + d["aux"] + d["verb"] + d["obj"])

# --- Transformations (each takes a structure, returns a NEW structure) ---
def T_do_support(d):
    '''English needs a dummy auxiliary 'do' before it can invert. Insert it if absent.'''
    d = {k: list(v) for k, v in d.items()}
    if not d["aux"]:
        d["aux"] = ["do"]
    return d

def T_subject_aux_inversion(d):
    '''Move the auxiliary to the front of the subject. THE question transformation.'''
    d = {k: list(v) for k, v in d.items()}
    if d["aux"]:
        d["subj"] = d["aux"] + d["subj"]
        d["aux"] = []
    return d

def T_2nd_person(d):
    '''The bot addresses the user: 'i' -> 'you'.'''
    d = {k: list(v) for k, v in d.items()}
    d["subj"] = ["you" if w == "i" else w for w in d["subj"]]
    return d

def apply_pipeline(d, rules):
    for r in rules:
        d = r(d)
    return d


In [14]:
DEEP = deep_structure(subj=["i"], verb=["want"], obj=["two", "dosas"])
print("deep structure        :", linearize(DEEP))

RIGHT = [T_2nd_person, T_do_support, T_subject_aux_inversion]
WRONG = [T_2nd_person, T_subject_aux_inversion, T_do_support]   # inversion BEFORE do-support

print("correct rule order    :", linearize(apply_pipeline(DEEP, RIGHT)), "?")
print("wrong rule order      :", linearize(apply_pipeline(DEEP, WRONG)), "?   <-- garbage")
print()
print("Same rules. Different order. This is what 'transformations work like a program' means.")


deep structure        : i want two dosas
correct rule order    : do you want two dosas ?
wrong rule order      : you do want two dosas ?   <-- garbage

Same rules. Different order. This is what 'transformations work like a program' means.


In [15]:
# ==== TODO (TEAM) ====
# 1. Write confirm_en() for YOUR domain using the pipeline above.
# 2. Write confirm_l2(): produce the same yes-no question in your regional language.
#    Do NOT copy subject-aux inversion. Use whatever your language actually does
#    (question particle, clitic, intonation marker, verb morphology).

def confirm_en(subj, verb, obj):
    d = deep_structure(subj=subj, verb=verb, obj=obj)
    return linearize(apply_pipeline(d, [T_2nd_person, T_do_support, T_subject_aux_inversion])) + "?"

def confirm_l2(subj, verb, obj):
    # e.g. Tamil: append the yes-no particle instead of moving anything.
    return " ".join(subj + obj + verb) + " -aa?"      # <-- REPLACE with your language's real form

print("EN:", confirm_en(["i"], ["want"], ["two", "dosas"]))
print(f"{L2}:", confirm_l2(["neenga"], ["vendum"], ["rendu", "dosa"]))


EN: do you want two dosas?
Tamil: neenga rendu dosa vendum -aa?


In [17]:
# ==== TODO (TEAM) ====  Write 3-4 sentences. Marked.
OBSERVATION_TRANSFORMATION = '''
English forms a yes/no question by ...
<L2> forms a yes/no question by ...
Therefore a movement-based transformational grammar is a good/bad fit for <L2> because ...
'''
print(OBSERVATION_TRANSFORMATION)



English forms a yes/no question by ...
<L2> forms a yes/no question by ...
Therefore a movement-based transformational grammar is a good/bad fit for <L2> because ...



---
# Part 4 — Valencies and Interpretation
### Topic 4 · *The verb is the intent*

A verb opens a fixed number of **slots** (valencies), like an atom's bonds. *dar* (to give) is trivalent: giver, thing given, receiver. The slots that belong to the verb are **actants**; the optional circumstantial ones (time, place, manner) are **circonstants**.

The list of a verb's actants and how each is realised syntactically is its **subcategorization frame**.

Now the reveal. Here is a Dialogflow / Rasa / Alexa intent schema:

```json
{"intent": "order_food", "slots": [{"name": "item", "required": true}, ...]}
```

That **is** a subcategorization frame. Actants → required slots. Circonstants → optional entities. One verb with two frames → one intent with two forms. The industry rediscovered Tesnière and Fillmore and gave it a JSON serialisation.


In [ ]:
# ===== REFERENCE IMPLEMENTATION (canteen) — do not edit =====
CANTEEN_VALENCY = {
    "order": {
        "gloss": "X orders Y (for Z)",
        "cues": ["want", "need", "get", "give", "add", "order"],
        "actants": [
            {"n": 1, "role": "agent",       "realisation": "subject",      "required": True,  "slot": "customer"},
            {"n": 2, "role": "patient",     "realisation": "direct object", "required": True,  "slot": "item"},
            {"n": 3, "role": "beneficiary", "realisation": "'for' PP",     "required": False, "slot": "for_whom"},
        ],
        "circonstants": ["time", "table_number", "payment_mode"],
    },
    "cancel": {
        "gloss": "X cancels Y",
        "cues": ["cancel", "remove", "drop"],
        "actants": [
            {"n": 1, "role": "agent",   "realisation": "subject",       "required": True, "slot": "customer"},
            {"n": 2, "role": "patient", "realisation": "direct object", "required": True, "slot": "order_ref"},
        ],
        "circonstants": ["reason"],
    },
    # ONE VERB, TWO FRAMES  (cf. the two frames of Spanish 'dar' in the slides)
    "pay": {
        "gloss": "frame A: X pays Y[amount]   |   frame B: X pays Y[person] Z[amount]",
        "cues": ["pay", "paying"],
        "actants": [
            {"n": 1, "role": "agent",     "realisation": "subject",        "required": True,  "slot": "customer"},
            {"n": 2, "role": "recipient", "realisation": "indirect object", "required": False, "slot": "payee"},
            {"n": 3, "role": "amount",    "realisation": "direct object",  "required": True,  "slot": "amount"},
        ],
        "circonstants": ["payment_mode"],
    },
    "amend": {
        "gloss": "X amends Y[slot] to Z[value]  -- revises the CURRENT order, does not start a new one",
        "cues": ["make", "change", "instead"],
        "actants": [
            {"n": 1, "role": "agent", "realisation": "subject",       "required": True, "slot": "customer"},
            {"n": 2, "role": "theme", "realisation": "direct object", "required": True, "slot": "quantity"},
        ],
        "circonstants": [],
    },
    "price": {
        "gloss": "X asks the price of Y",
        "cues": ["price", "cost", "much", "rate"],
        "actants": [
            {"n": 1, "role": "agent",  "realisation": "subject",  "required": True, "slot": "customer"},
            {"n": 2, "role": "target", "realisation": "'of' PP",   "required": True, "slot": "item"},
        ],
        "circonstants": ["quantity"],
    },
    "avail": {
        "gloss": "X asks whether Y is available",
        "cues": ["available", "availability", "stock", "have"],
        "actants": [
            {"n": 1, "role": "agent",  "realisation": "subject", "required": True, "slot": "customer"},
            {"n": 2, "role": "target", "realisation": "subject of the predicate", "required": True, "slot": "item"},
        ],
        "circonstants": [],
    },
}


In [ ]:
def to_intent_schema(verb, spec):
    '''Subcategorization frame -> the JSON an intent-based dialogue platform eats.'''
    return {
        "intent": verb,
        "slots": [
            {"name": a["slot"], "semantic_role": a["role"], "required": a["required"],
             "realisation": a["realisation"]}
            for a in spec["actants"]
        ],
        "entities": spec["circonstants"],
    }

print(json.dumps(to_intent_schema("order", CANTEEN_VALENCY["order"]), indent=2))


In [ ]:
def valency_report(valency):
    print(f"{'verb':<10}{'#actants':>9}{'#required':>11}{'#circ':>7}   frame")
    print("-" * 78)
    for v, spec in valency.items():
        req = sum(1 for a in spec["actants"] if a["required"])
        print(f"{v:<10}{len(spec['actants']):>9}{req:>11}{len(spec['circonstants']):>7}   {spec['gloss']}")

valency_report(CANTEEN_VALENCY)


In [ ]:
# ==== TODO (TEAM) ====
# Build VALENCY for 5 verbs from YOUR domain. Rules:
#   * at least one verb must be trivalent (3 actants)
#   * at least one verb must have TWO different frames — say so in the gloss
#   * every actant needs a semantic role AND a syntactic realisation
#   * circonstants are NOT actants. If it can be dropped without the sentence
#     becoming incomplete, it is a circonstant.

VALENCY = {
    # "book": {"gloss": "...", "actants": [...], "circonstants": [...]},
}

if not VALENCY:
    VALENCY = CANTEEN_VALENCY
    print("!! Using the reference valency table. Replace it.")

valency_report(VALENCY)
SCHEMAS = {v: to_intent_schema(v, s) for v, s in VALENCY.items()}
print()
print("Generated", len(SCHEMAS), "intent schemas:", list(SCHEMAS))


---
# Part 5 — Constraints
### Topic 5 · *Break it, then fix it — and count the rules*

Bring back your overgeneration attacks from Part 2. Plain CFG rules cannot block them, because `NP -> NUM N` has no idea that `two` is plural and `dosa` is singular.

The fix from the slides: put **features** on constituents and write the rule as an **equation** — the same variable `?n` on both daughters, so they must unify.

The point is not that it works. The point is **how many rules it saves.** Run the blow-up counter and look at the number.


In [ ]:
# --- A: without features. Number agreement by brute force. ---
UNFEATURED = CFG.fromstring('''
    S       -> WANT NP_SG | WANT NP_PL
    WANT    -> 'i' 'want'
    NP_SG   -> QNT_SG N_SG
    NP_PL   -> QNT_PL N_PL
    QNT_SG  -> 'one' | 'a'
    QNT_PL  -> 'two' | 'three'
    N_SG    -> 'dosa' | 'idli'
    N_PL    -> 'dosas' | 'idlis'
''')
p = ChartParser(UNFEATURED)
for s in ["i want two dosas", "i want two dosa", "i want a dosa", "i want a dosas"]:
    ok = bool(list(p.parse(tokens(s))))
    print(f"  {'OK ' if ok else 'BAD'}  {s}")


In [ ]:
# --- B: with features. ONE NP rule, and an equation. ---
FEATURED = FeatureGrammar.fromstring('''
    % start S
    S            -> WANT NP[NUM=?n]
    NP[NUM=?n]   -> QNT[NUM=?n] N[NUM=?n]
    WANT         -> 'i' 'want'
    QNT[NUM=sg]  -> 'one' | 'a'
    QNT[NUM=pl]  -> 'two' | 'three'
    N[NUM=sg]    -> 'dosa' | 'idli'
    N[NUM=pl]    -> 'dosas' | 'idlis'
''')
fp = FeatureChartParser(FEATURED)
for s in ["i want two dosas", "i want two dosa", "i want a dosa", "i want a dosas"]:
    ok = bool(list(fp.parse(tokens(s))))
    print(f"  {'OK ' if ok else 'BAD'}  {s}")


In [ ]:
# --- The rule-count blow-up. This is the whole argument for features. ---
def blowup(base_phrase_rules, feature_dims):
    '''feature_dims: {'NUM': 2, 'GEN': 3, ...}  How many flat rules would you need?'''
    combos = 1
    for v in feature_dims.values():
        combos *= v
    flat = base_phrase_rules * combos
    print(f"  features   : {feature_dims}")
    print(f"  combos     : {combos}")
    print(f"  flat rules : {base_phrase_rules} phrase rules x {combos} = {flat}")
    print(f"  featured   : {base_phrase_rules}  (one rule per phrase type, with equations)")
    print(f"  saving     : {flat - base_phrase_rules} rules\n")

print("English (number only):");            blowup(4, {"NUM": 2})
print("Spanish (number + gender):");        blowup(4, {"NUM": 2, "GEN": 2})
print("Hindi   (number + gender + person + honorific):")
blowup(4, {"NUM": 2, "GEN": 2, "PER": 3, "HON": 2})


In [ ]:
# ==== TODO (TEAM) ====
# 1. Write a FeatureGrammar for your domain that blocks your Part 2 overgeneration attacks.
# 2. Pick the feature set your REGIONAL language actually needs and run blowup() on it.
#    Hindi/Marathi: NUM, GEN, PER, HON. Tamil: NUM, PER, GEN(3-way), HON. Malayalam: NUM, PER, HON.

MY_FEATURED = FeatureGrammar.fromstring('''
    % start S
    S          -> WANT NP[NUM=?n]
    NP[NUM=?n] -> QNT[NUM=?n] N[NUM=?n]
    WANT       -> 'i' 'want'
    QNT[NUM=sg]-> 'one'
    QNT[NUM=pl]-> 'two'
    N[NUM=sg]  -> 'thing'
    N[NUM=pl]  -> 'things'
''')

my_fp = FeatureChartParser(MY_FEATURED)
for bad in ATTACKS["overgeneration"] or ["i want two thing"]:
    blocked = not list(my_fp.parse(tokens(bad)))
    print(f"  {'BLOCKED' if blocked else 'STILL ACCEPTED'} : {bad}")

print()
print(f"Feature dimensions needed for {L2}:")
blowup(4, {"NUM": 2, "PER": 3})     # <-- replace with your language's real feature set


---
# Part 6 — HPSG and Unification
### Topic 6 · *Dialogue state tracking IS unification*

HPSG replaces the transformation machinery with two ideas:
1. **The head principle** — every constituent has a *head daughter*, and the mother inherits the head's features. `el muchacho` is an NP because *muchacho* is a noun; `quisiera` heads the VP.
2. **Unification** — two feature sets combine iff they agree on every shared feature. Conflict = failure, and the *specific conflicting feature* is the diagnosis.

Now the reveal. A multi-turn dialogue is exactly this. Turn 1 gives you `{item: dosa}`. Turn 2 gives you `{quantity: 2}`. Combining them is unification. Turn 3 says `no, make it three` — that **conflicts** with `quantity: 2`, and how your bot handles that conflict is the entire difference between a usable assistant and a frustrating one.


In [ ]:
def unify(a, b, path=""):
    '''Unify two feature structures (nested dicts).
       Returns (result, conflicts). conflicts is [] iff unification succeeded.'''
    result, conflicts = dict(a), []
    for k, vb in b.items():
        here = f"{path}.{k}" if path else k
        if k not in result:
            result[k] = vb
        elif isinstance(result[k], dict) and isinstance(vb, dict):
            sub, subc = unify(result[k], vb, here)
            result[k] = sub
            conflicts += subc
        elif result[k] != vb:
            conflicts.append({"feature": here, "existing": result[k], "incoming": vb})
    return result, conflicts


# The unification card game, in code.
tests = [
    ({"cat": "N", "num": "pl", "gen": "fem"}, {"cat": "N", "num": "pl"}),
    ({"cat": "N", "num": "sg"},               {"cat": "N", "num": "pl"}),
    ({"cat": "Det", "num": "sg", "gen": "masc"}, {"cat": "Det", "gen": "fem"}),
]
for a, b in tests:
    r, c = unify(a, b)
    print(f"  {a}\n+ {b}\n= {r if not c else 'FAIL ' + str(c)}\n")


In [ ]:
def combine(head, dependent, mother_cat=None):
    '''HPSG head principle: the mother inherits the HEAD's features,
       after the head and dependent unify on their shared features.'''
    merged, conflicts = unify(head, dependent)
    if conflicts:
        return None, conflicts
    mother = dict(merged)
    mother["cat"] = mother_cat or head["cat"]      # category comes from the HEAD
    mother["head"] = head.get("lex", head["cat"])
    return mother, []

# 'el muchacho'  (from the slides)
el       = {"cat": "Det", "lex": "el",       "num": "sg", "gen": "masc"}
muchacho = {"cat": "N",   "lex": "muchacho", "num": "sg", "gen": "masc"}
print("el muchacho    ->", combine(muchacho, el, mother_cat="NP")[0])

# 'las estudiantes'  — estudiante is gender-underspecified, so it takes the determiner's
las         = {"cat": "Det", "lex": "las",         "num": "pl", "gen": "fem"}
estudiantes = {"cat": "N",   "lex": "estudiantes", "num": "pl"}
print("las estudiantes->", combine(estudiantes, las, mother_cat="NP")[0])

# conflict
el_bad = {"cat": "Det", "lex": "el", "num": "sg", "gen": "masc"}
print("el estudiantes ->", combine(estudiantes, el_bad, mother_cat="NP")[1])


In [ ]:
class DialogueState:
    '''Multi-turn slot filling. Every turn is a unification against the running frame.'''

    def __init__(self, schema):
        self.schema = schema                 # an intent schema from Part 4
        self.frame = {}
        self.log = []

    def update(self, new_slots, allow_override=False):
        merged, conflicts = unify(self.frame, new_slots)
        if conflicts and not allow_override:
            self.log.append(("CONFLICT", conflicts))
            return {"status": "conflict", "conflicts": conflicts}
        if conflicts and allow_override:
            merged = {**self.frame, **new_slots}     # repair: newest wins
            self.log.append(("OVERRIDE", conflicts))
        else:
            self.log.append(("MERGE", new_slots))
        self.frame = merged
        return {"status": "ok", "frame": self.frame}

    def missing(self):
        return [s["name"] for s in self.schema["slots"]
                if s["required"] and s["name"] not in self.frame]

    def complete(self):
        return not self.missing()


In [ ]:
st = DialogueState(to_intent_schema("order", CANTEEN_VALENCY["order"]))

print("U: i want dosa")
print("  ", st.update({"item": "dosa"}), "| still missing:", st.missing())

print("U: make it two")
print("  ", st.update({"quantity": "two"}), "| still missing:", st.missing())

print("U: no, make it three          <-- CONFLICT with quantity=two")
r = st.update({"quantity": "three"})
print("  ", r)

print("   repair strategy: newest turn wins")
print("  ", st.update({"quantity": "three"}, allow_override=True))
print("   complete?", st.complete(), "| frame:", st.frame)


In [ ]:
# ==== TODO (TEAM) ====
# 1. Run a 4-turn dialogue for YOUR domain through DialogueState, including one conflict.
# 2. Answer in ANSWER_REPAIR: when the user contradicts an already-filled slot, should the
#    bot (a) silently overwrite, (b) ask for confirmation, or (c) reject? Justify in terms
#    of unification failure -- what does the conflict record actually tell you?

ANSWER_REPAIR = '''
...
'''


---
# Part 7 — Meaning-Text Theory: Government Patterns
### Topic 7 · *The GP is your response generator*

MTT models language as a **multistage transformer** — meaning ⇄ deep syntax ⇄ surface syntax ⇄ morphology ⇄ text — with no transformations, only inter-level correspondences.

A **government pattern** is the table that says, for one lexeme, how each *semantic* valency is realised on the surface: which preposition, which case, which word order, which combinations are legal.

A GP is strictly richer than a subcategorization frame (Part 4). The frame says *"actant 2 is a direct object."* The GP says *"actant 2 is a direct object, and in Tamil it takes the accusative `-ai`, and if actant 3 is present the order is 1-3-2-V."* That extra information is precisely what you need to **generate** text — which is why this cell is your bot's response generator, not just another table.


In [ ]:
# ===== REFERENCE IMPLEMENTATION (canteen) — do not edit =====
# order[key]  ->  sequence of semantic valency numbers + "V" for the verb
# marks[key]  ->  the case marker / postposition / preposition attached to that valency
CANTEEN_GP = {
    "order": {
        "en": {"verb": "ordered", "order": ["1", "V", "2", "3"],
               "marks": {"1": "", "2": "", "3": "for "}},
        "l2": {"verb": "order pannineenga", "order": ["1", "3", "2", "V"],
               "marks": {"1": "", "2": "-ai", "3": "-ukku"}},
    },
    "cancel": {
        "en": {"verb": "cancelled", "order": ["1", "V", "2"],
               "marks": {"1": "", "2": ""}},
        "l2": {"verb": "cancel pannineenga", "order": ["1", "2", "V"],
               "marks": {"1": "", "2": "-ai"}},
    },
    "price": {
        "en": {"verb": "want the price of", "order": ["1", "V", "2"],
               "marks": {"1": "", "2": ""}},
        "l2": {"verb": "vilai kekkireenga", "order": ["1", "2", "V"],
               "marks": {"1": "", "2": "-oda"}},
    },
    "avail": {
        "en": {"verb": "are asking about", "order": ["1", "V", "2"],
               "marks": {"1": "", "2": ""}},
        "l2": {"verb": "irukka-nu kekkireenga", "order": ["1", "2", "V"],
               "marks": {"1": "", "2": ""}},
    },
}
# NOTE: the 'l2' rows above are ILLUSTRATIVE PLACEHOLDERS in rough transliteration.
# Replace them with forms verified by the fluent speaker on your team. Marked.


In [ ]:
def realise(gp, lang, filled):
    '''Government pattern + a filled semantic frame  ->  a surface string.
       filled: {"1": "you", "2": "two dosas", "3": "table 4"}  (missing keys are skipped)

       A mark written "-xx" is a SUFFIX (a case marker: it attaches to the end of the
       group). Anything else is a PREPOSITION and precedes it. That distinction is
       exactly the typological difference between English and most Indian languages.'''
    spec = gp[lang]
    out = []
    for k in spec["order"]:
        if k == "V":
            out.append(spec["verb"])
        elif k in filled:
            mark, val = spec["marks"].get(k, ""), str(filled[k])
            out.append(val + mark if mark.startswith("-") else mark + val)
    return " ".join(out)


frame = {"1": "you", "2": "two dosas"}
print("EN  :", realise(CANTEEN_GP["order"], "en", frame))
print("L2  :", realise(CANTEEN_GP["order"], "l2", {"1": "neenga", "2": "rendu dosa"}))
print()
frame3 = {"1": "you", "2": "two dosas", "3": "table 4"}
print("EN 3:", realise(CANTEEN_GP["order"], "en", frame3))
print("L2 3:", realise(CANTEEN_GP["order"], "l2", {"1": "neenga", "2": "rendu dosa", "3": "table 4"}))
print()
print("ONE semantic frame -> TWO surface languages. That is a multistage transformer.")


In [ ]:
# ==== TODO (TEAM) ====
# Build GP for 3 verbs of your domain, in English and your regional language.
# Then answer GP_VS_FRAME.

GP = {
    # "book": {"en": {...}, "l2": {...}},
}
if not GP:
    GP = CANTEEN_GP
    print("!! Using the reference GP table. Replace it (and fix the L2 forms).")

GP_VS_FRAME = '''
The subcategorization frame in Part 4 tells us ...
The government pattern additionally tells us ...
The information we could NOT have generated <L2> output without is ...
'''
print(GP_VS_FRAME)


---
# Part 8 — Dependency Trees and Semantic Links
### Topic 8 · *Hand parser vs machine parser*

Constituency groups words into phrases. **Dependency** links each word directly to its governor — no phrase nodes at all. The verb is the root; its actants hang off it as labelled arcs.

That maps onto your bot far more directly than a constituency tree: **root verb = intent, `nsubj`/`obj`/`nummod` = slots.** And it needs no grammar written by hand.

Do this in three steps and *keep the numbers* — they go in your Part 9 write-up:
1. Hand-draw dependency trees for 5 utterances. Type them in as `(dependent, head)` pairs.
2. Run spaCy on the same 5. Compute UAS (unlabelled attachment score) against your hand trees.
3. **Argue.** Where you and the parser disagree, decide who is right and say why. Universal Dependencies annotators disagree about these too.


In [ ]:
%pip install -q spacy


In [ ]:
!python -m spacy download en_core_web_sm


In [ ]:
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    SPACY_OK = True
except Exception as e:
    SPACY_OK = False
    print("spaCy unavailable:", e)
    print("Restart the runtime (Colab: Runtime > Restart) and re-run the two cells above.")

if SPACY_OK:
    doc = nlp("i want two masala dosas")
    print(f"{'TOKEN':<10}{'POS':<8}{'DEP':<12}{'HEAD'}")
    for t in doc:
        print(f"{t.text:<10}{t.pos_:<8}{t.dep_:<12}{t.head.text}")


In [ ]:
if SPACY_OK:
    from spacy import displacy
    displacy.render(nlp("i want two masala dosas"), style="dep", jupyter=True,
                    options={"compact": True, "distance": 90})


In [ ]:
# ==== TODO (TEAM) ====  Your HAND-DRAWN trees, as (dependent_index, head_index) pairs.
# Index words from 0. Use -1 as the head of the ROOT.
#   "i want two dosas"  ->  i(0)<-want(1), want(1)=ROOT, two(2)<-dosas(3), dosas(3)<-want(1)
HAND_TREES = {
    "i want two dosas": [(0, 1), (1, -1), (2, 3), (3, 1)],
    # add 4 more from YOUR test set
}

def uas(hand_heads, doc):
    '''Unlabelled attachment score: fraction of tokens whose HEAD you and spaCy agree on.'''
    pred = {t.i: (-1 if t.head.i == t.i else t.head.i) for t in doc}
    agree = sum(1 for dep, head in hand_heads if pred.get(dep) == head)
    return agree, len(hand_heads), [(dep, head, pred.get(dep))
                                    for dep, head in hand_heads if pred.get(dep) != head]

if SPACY_OK:
    tot_a = tot_n = 0
    for sent, hand in HAND_TREES.items():
        a, n, diffs = uas(hand, nlp(sent))
        tot_a += a; tot_n += n
        print(f"{sent!r:<40} UAS {a}/{n}")
        for d, mine, theirs in diffs:
            print(f"    token {d}: you say head={mine}, spaCy says head={theirs}  <-- ARGUE")
    if tot_n:
        print(f"\nOVERALL UAS = {tot_a}/{tot_n} = {tot_a/tot_n:.0%}")


In [ ]:
# --- Dependency-based slot extraction: the CFG's replacement, in 15 lines ---
NUMWORDS = {"one": "one", "two": "two", "three": "three", "a": "one", "an": "one",
            "1": "one", "2": "two", "3": "three"}

def extract_slots_dep(doc):
    frame = {}
    root = next((t for t in doc if t.head is t), None)
    if root is not None:
        frame["_verb"] = root.lemma_
    for t in doc:
        if t.dep_ in ("dobj", "obj", "attr", "pobj"):
            frame["item"] = t.lemma_
            for c in t.children:
                if c.dep_ == "nummod" or c.lower_ in NUMWORDS:
                    frame["quantity"] = NUMWORDS.get(c.lower_, c.text)
                elif c.dep_ in ("amod", "compound"):
                    frame["modifier"] = c.lemma_
            break
    return frame

if SPACY_OK:
    for u in CANTEEN_TESTSET[:6]:
        print(f"{u['text']:<32} -> {extract_slots_dep(nlp(u['text']))}")


---
# Part 9 — Capstone: assemble, evaluate, compare

Everything above becomes one agent:

```
utterance
   -> CFG / dependency parse        (Parts 2, 8)
   -> semantic frame                (Part 4 intent schema)
   -> unify into dialogue state     (Part 6)
   -> required slots filled?  no -> ask     yes -> confirm
   -> realise via government pattern, EN + L2   (Part 7)
```

Then you measure it, three ways:
1. **Accuracy** on the test set — intent accuracy and slot F1.
2. **Paraphrase invariance** — 10 different surface forms of the *same* meaning. Does the bot map them all to one frame? This is MTT's *not-uniqueness of meaning→text mapping* (Unit 4), tested empirically.
3. **Head to head against an LLM** on the adversarial test set another team wrote for you.


In [ ]:
class GrammarBot:
    def __init__(self, cfg, valency, gp, l2_name="L2", use_dep=False):
        self.parser = ChartParser(cfg)
        self.valency = valency
        self.gp = gp
        self.l2_name = l2_name
        self.use_dep = use_dep
        self.state = None
        self.intent = None

    # --- understanding ---
    def parse_frame(self, text):
        toks = tokens(text)
        if self.use_dep and SPACY_OK:
            f = extract_slots_dep(nlp(text))
            verb = f.pop("_verb", None)
            intent = verb if verb in self.valency else self._guess_intent(toks)
            return intent, f
        try:
            trees = list(self.parser.parse(toks))
        except ValueError:
            trees = []
        if not trees:
            return None, {}
        return self._guess_intent(toks), self._slots_from_tree(trees[0])

    def _guess_intent(self, toks):
        '''The verb identifies the intent (Part 4). Match on the cue words the
           valency table declares, longest-specific first.'''
        for v, spec in self.valency.items():
            if set(spec.get("cues", [v])) & set(toks):
                return v
        return list(self.valency)[0]

    def _slots_from_tree(self, tree):
        slots = {}
        for st in tree.subtrees():
            if st.label() == "NUM":
                slots["quantity"] = st[0]
            elif st.label() == "N":
                slots.setdefault("item", st[0].rstrip("s"))
            elif st.label() == "ADJ":
                slots["modifier"] = st[0]
        return slots

    # --- the turn loop ---
    def turn(self, text):
        intent, slots = self.parse_frame(text)
        if intent is None:
            return "Sorry, I did not understand that."
        # An AMEND revises the order in progress -- it must NOT open a new frame,
        # or the item filled two turns ago is lost. This is the whole reason
        # dialogue state is a unification store and not a fresh parse each turn.
        if intent == "amend" and self.state is not None:
            intent = self.intent
        if self.state is None or intent != self.intent:
            self.intent = intent
            self.state = DialogueState(to_intent_schema(intent, self.valency[intent]))
        res = self.state.update(slots, allow_override=True)
        missing = [m for m in self.state.missing() if m not in ("customer",)]
        if missing:
            return f"Which {missing[0]}?"
        return self.confirm()

    # --- generation, via the government pattern ---
    def confirm(self):
        f = self.state.frame
        obj = " ".join(str(f[k]) for k in ("quantity", "modifier", "item") if k in f)
        sem = {"1": "you", "2": obj}
        gp = self.gp.get(self.intent, list(self.gp.values())[0])
        return (f"EN : {realise(gp, 'en', sem)}?\n"
                f"{self.l2_name:<3}: {realise(gp, 'l2', sem)}?")


bot = GrammarBot(CANTEEN_CFG, CANTEEN_VALENCY, CANTEEN_GP, l2_name=L2)
for line in ["i want two dosas", "make it three"]:
    print("U:", line)
    print("B:", bot.turn(line))
    print()


In [ ]:
# --- Evaluation harness ---
def evaluate(bot, testset):
    intent_ok, tp, fp, fn = 0, 0, 0, 0
    for u in testset:
        pred_intent, pred_slots = bot.parse_frame(u["text"])
        if pred_intent == u["intent"]:
            intent_ok += 1
        gold = u["slots"]
        for k, v in pred_slots.items():
            if k in gold and str(gold[k]) == str(v):
                tp += 1
            else:
                fp += 1
        for k in gold:
            if k not in pred_slots:
                fn += 1
    n = len(testset)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"intent_accuracy": intent_ok / n, "slot_precision": prec,
            "slot_recall": rec, "slot_f1": f1, "n": n}


cfg_bot = GrammarBot(CANTEEN_CFG, CANTEEN_VALENCY, CANTEEN_GP, use_dep=False)
print("CFG-based        :", evaluate(cfg_bot, CANTEEN_TESTSET))
if SPACY_OK:
    dep_bot = GrammarBot(CANTEEN_CFG, CANTEEN_VALENCY, CANTEEN_GP, use_dep=True)
    print("dependency-based :", evaluate(dep_bot, CANTEEN_TESTSET))
    print("\nWhich wins on recall? Which wins on precision? WHY? -> your write-up.")


In [ ]:
# --- Paraphrase invariance: 10 surface forms, ONE meaning (MTT) ---
PARAPHRASES = [
    "i want two dosas",
    "i would like two dosas",
    "give me two dosas",
    "can i get two dosas",
    "two dosas please",
    "i need two dosas",
    "make it two dosas",
    "could you get me two dosas",
    "2 dosas",
    "dosa two plates",
]
TARGET = {"item": "dosa", "quantity": "two"}

def paraphrase_test(bot, paras, target):
    hits = 0
    for p in paras:
        _, slots = bot.parse_frame(p)
        match = all(str(slots.get(k)) == str(v) for k, v in target.items())
        hits += match
        print(f"  {'OK ' if match else 'MISS'}  {p:<32} -> {slots}")
    print(f"\nparaphrase invariance = {hits}/{len(paras)} = {hits/len(paras):.0%}")
    return hits / len(paras)

paraphrase_test(cfg_bot, PARAPHRASES, TARGET)


> **A low score here is the expected result, not a bug.** The reference grammar scores about
> 40%: it misses `i would like`, `two dosas please`, `2 dosas`, `dosa two plates`. Each miss is
> one surface form of a meaning the grammar already handles — this is precisely MTT's
> *not-uniqueness of the meaning→text mapping*, and a hand-written CFG has to enumerate every
> branch of it by hand.
>
> Two things to do rather than patching rules until it hits 100%:
> 1. Count how many **new rules** it costs to buy each extra paraphrase. Plot that. It is the
>    real cost curve of rationalist NLP.
> 2. Run the same 10 through the dependency extractor and the LLM below. Report all three
>    numbers. That comparison, not the absolute score, is what Part 9 is marked on.


### The LLM baseline (optional, needs an API key)

Same task, no grammar. Set `ANTHROPIC_API_KEY` in your environment (Colab: the 🔑 Secrets panel, or `os.environ[...] = ...` in a cell you delete before submitting — **never commit a key**).

Run the same test set through both and compare. The interesting result is *not* which score is higher; it is **which errors each one makes**.


In [ ]:
%pip install -q anthropic


In [ ]:
import os

LLM_ON = bool(os.environ.get("ANTHROPIC_API_KEY"))

if LLM_ON:
    from anthropic import Anthropic
    client = Anthropic()

    LLM_SYSTEM = (
        "You are the natural-language understanding module of a " + DOMAIN + " assistant.\n"
        "Given one user utterance, return ONLY a JSON object, no prose and no code fences:\n"
        '{"intent": "<one of ' + str(INTENTS) + '>", "slots": {"item": "...", "quantity": "..."}}\n'
        "Omit any slot the utterance does not fill. Use singular lemmas for item, "
        "and English number words (one/two/three) for quantity."
    )

    def llm_parse(text):
        r = client.messages.create(
            model="claude-opus-5",
            max_tokens=4096,
            system=LLM_SYSTEM,
            messages=[{"role": "user", "content": text}],
        )
        if r.stop_reason == "refusal":
            return {"intent": None, "slots": {}}
        out = "".join(b.text for b in r.content if b.type == "text").strip()
        out = re.sub(r"^```(?:json)?|```$", "", out, flags=re.M).strip()
        return json.loads(out)
        # More robust alternative: pass output_config={"format": {"type": "json_schema",
        # "schema": {...}}} to guarantee schema-valid JSON instead of parsing free text.

    print(llm_parse("2 dosas"))
    print(llm_parse("dosa two plates"))
else:
    print("No ANTHROPIC_API_KEY set. Skipping the LLM baseline — the rest of the notebook still works.")


In [ ]:
def evaluate_llm(testset):
    intent_ok, tp, fp, fn = 0, 0, 0, 0
    for u in testset:
        try:
            pred = llm_parse(u["text"])
        except Exception as e:
            print("  parse error on", u["text"], "->", e)
            fn += len(u["slots"]); continue
        if pred.get("intent") == u["intent"]:
            intent_ok += 1
        gold, ps = u["slots"], pred.get("slots", {})
        for k, v in ps.items():
            if k in gold and str(gold[k]) == str(v): tp += 1
            else: fp += 1
        for k in gold:
            if k not in ps: fn += 1
    n = len(testset)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"intent_accuracy": intent_ok / n, "slot_precision": prec,
            "slot_recall": rec, "slot_f1": f1, "n": n}

if LLM_ON:
    print("grammar bot :", evaluate(cfg_bot, CANTEEN_TESTSET))
    print("LLM         :", evaluate_llm(CANTEEN_TESTSET))


In [ ]:
# ==== TODO (TEAM) ====  ADVERSARIAL SET
# You WRITE this for another team and RECEIVE theirs. 50 utterances that are legal in the
# domain but designed to break a hand-written grammar: ellipsis, code-mixing, typos,
# unusual word order, negation, multi-intent turns.
ADVERSARIAL_FOR_OTHER_TEAM = [
    # {"text": "...", "intent": "...", "slots": {...}},
]
RECEIVED_ADVERSARIAL = [
    # paste the set the other team gives you
]

if RECEIVED_ADVERSARIAL:
    print("grammar bot :", evaluate(cfg_bot, RECEIVED_ADVERSARIAL))
    if LLM_ON:
        print("LLM         :", evaluate_llm(RECEIVED_ADVERSARIAL))


---
# Write-up (4 of the 20 marks)

Answer with **numbers from your own runs**, not opinions. One paragraph each.

1. **Coverage vs. rules.** What was your final grammar-golf score, and which rule bought you the most coverage per rule added?
2. **Overgeneration.** How many nonsense strings did your Part 2 CFG accept? How many did features block in Part 5? What still gets through, and would *more* features fix it, or is it a semantic problem a grammar cannot see?
3. **Frame vs. schema.** Show one verb where your subcategorization frame and your intent schema are literally the same object. Then show one thing the government pattern knows that the schema does not.
4. **Unification.** Report your conflict-repair strategy and one dialogue where it produced the wrong answer.
5. **Hand parse vs machine parse.** Report your UAS. Pick the one disagreement where you are confident *spaCy* is wrong, and defend it.
6. **CFG vs dependency vs LLM.** Give the three score triples. Then the real question: **name one thing your grammar does that the LLM gets wrong, and one thing the LLM does that you could never hand-write.**
7. **The regional language.** Which single component of Unit 2 (CFG / transformations / features / GP / dependency) transferred worst from English to your language, and why?


In [ ]:
# --- Submission summary. Run this last; paste the output into your report. ---
summary = {
    "team": TEAM,
    "domain": DOMAIN,
    "l2": L2,
    "testset_size": len(TESTSET),
    "cfg_rules": len(MY_CFG.productions()),
    "cfg_coverage": coverage(MY_CFG, TESTSET, verbose=False)[0] / len(TESTSET),
    "attacks_found": {k: len(v) for k, v in ATTACKS.items()},
    "verbs_with_valency": list(VALENCY),
    "gp_verbs": list(GP),
    "spacy_available": SPACY_OK,
    "llm_baseline_run": LLM_ON,
}
print(json.dumps(summary, indent=2))
